# Step 1: Preprocessing & CRS Reprojection

**Contract:** every downstream phase assumes ETRS89 / UTM 29N (**EPSG:25829**).

This notebook builds a synthetic WGS84 scene over the Wicklow Mountains (a stand-in for a raw Sentinel-2 tile) and pushes it through `src/crs_handler.py`.

In [ ]:
import os
import sys

import numpy as np
import rasterio
from rasterio.transform import from_bounds

sys.path.append(os.path.abspath('../src'))
from crs_handler import reproject_to_utm29n

os.makedirs('../data/input', exist_ok=True)
os.makedirs('../data/output', exist_ok=True)

rng = np.random.default_rng(42)
src_path = '../data/input/demo_wgs84.tif'
profile = {"driver": "GTiff", "height": 200, "width": 200, "count": 1, "dtype": "float32",
           "crs": "EPSG:4326", "transform": from_bounds(-6.4, 52.95, -6.1, 53.15, 200, 200)}
with rasterio.open(src_path, 'w', **profile) as dst:
    dst.write((rng.random((200, 200)) * 0.8 + 0.1).astype('float32'), 1)

out_path = reproject_to_utm29n(src_path, '../data/output/01_reprojected.tif')

with rasterio.open(src_path) as a, rasterio.open(out_path) as b:
    print(f"BEFORE: {a.crs}  bounds={tuple(round(v, 3) for v in a.bounds)}")
    print(f"AFTER : {b.crs}  bounds={tuple(round(v, 1) for v in b.bounds)}")
    assert b.crs.to_epsg() == 25829, "CRS contract violated!"

print("✅ CRS contract enforced: EPSG:25829 (ETRS89 / UTM 29N)")